# 🗄️ Notebook 2 — SQL Analysis with SQLite

**Goal:** Use SQL to answer core business questions on the cleaned dataset.

---
### Queries Covered
- KPIs: Total Revenue, Total Orders, Average Order Value
- Monthly revenue trend
- Top 10 products by revenue
- Revenue by country
- Top 10 customers by spend
- Create `rfm_analysis` table (Recency, Frequency, Monetary)
- Customer revenue ranking with window functions

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load cleaned data into SQLite (in-memory or file)
conn = sqlite3.connect('ecommerce.db')
df = pd.read_csv('cleaned_ecommerce.csv')
df.to_sql('ecommerce_sales', conn, if_exists='replace', index=False)
print(f'✅ Loaded {len(df):,} rows into SQLite table: ecommerce_sales')

## 📊 KPI 1: Total Revenue

In [ ]:
result = pd.read_sql("""
    SELECT
        ROUND(SUM(Revenue), 2)                          AS Total_Revenue,
        COUNT(DISTINCT Invoice)                         AS Total_Orders,
        ROUND(SUM(Revenue) * 1.0 / COUNT(DISTINCT Invoice), 2) AS Avg_Order_Value,
        COUNT(DISTINCT [Customer ID])                   AS Total_Customers
    FROM ecommerce_sales
""", conn)
result

## 📅 Monthly Revenue Trend

In [ ]:
monthly = pd.read_sql("""
    SELECT
        MonthYear,
        ROUND(SUM(Revenue), 2) AS Revenue
    FROM ecommerce_sales
    GROUP BY MonthYear
    ORDER BY MonthYear
""", conn)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(monthly))
ax.fill_between(x, monthly['Revenue'], alpha=0.15, color='#2563EB')
ax.plot(x, monthly['Revenue'], color='#2563EB', linewidth=2.5, marker='o', markersize=5)
ax.set_xticks(x)
ax.set_xticklabels(monthly['MonthYear'], rotation=45, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v/1e6:.1f}M'))
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏆 Top 10 Products by Revenue

In [ ]:
top_products = pd.read_sql("""
    SELECT
        Description,
        ROUND(SUM(Revenue), 2) AS Total_Revenue,
        SUM(Quantity)          AS Total_Units_Sold
    FROM ecommerce_sales
    GROUP BY Description
    ORDER BY Total_Revenue DESC
    LIMIT 10
""", conn)
top_products

## 🌍 Revenue by Country

In [ ]:
by_country = pd.read_sql("""
    SELECT
        Country,
        ROUND(SUM(Revenue), 2)                                          AS Total_Revenue,
        ROUND(SUM(Revenue) * 100.0 / (SELECT SUM(Revenue) FROM ecommerce_sales), 1) AS Revenue_Pct
    FROM ecommerce_sales
    GROUP BY Country
    ORDER BY Total_Revenue DESC
    LIMIT 10
""", conn)
by_country

## 👤 Top 10 Customers by Spend

In [ ]:
top_customers = pd.read_sql("""
    SELECT
        [Customer ID]               AS CustomerID,
        ROUND(SUM(Revenue), 2)      AS Total_Spend,
        COUNT(DISTINCT Invoice)     AS Orders,
        ROUND(AVG(Revenue), 2)      AS Avg_Order_Value
    FROM ecommerce_sales
    GROUP BY [Customer ID]
    ORDER BY Total_Spend DESC
    LIMIT 10
""", conn)
top_customers

## 📦 Create RFM Analysis Table

In [ ]:
conn.execute('DROP TABLE IF EXISTS rfm_analysis')
conn.execute("""
    CREATE TABLE rfm_analysis AS
    SELECT
        [Customer ID] AS CustomerID,
        CAST(
            julianday((SELECT MAX(InvoiceDate) FROM ecommerce_sales))
            - julianday(MAX(InvoiceDate))
        AS INTEGER) AS Recency,
        COUNT(DISTINCT Invoice) AS Frequency,
        ROUND(SUM(Revenue), 2)  AS Monetary
    FROM ecommerce_sales
    GROUP BY [Customer ID]
""")
conn.commit()

rfm = pd.read_sql('SELECT * FROM rfm_analysis', conn)
print(f'RFM table: {len(rfm):,} customers')
rfm.head(10)

## 🏅 Customer Revenue Ranking (Window Function)

In [ ]:
ranked = pd.read_sql("""
    SELECT
        CustomerID,
        Monetary,
        RANK() OVER (ORDER BY Monetary DESC) AS Revenue_Rank
    FROM rfm_analysis
    LIMIT 20
""", conn)
ranked

In [ ]:
conn.close()
print('✅ SQL analysis complete. Database connection closed.')